<a href="https://colab.research.google.com/github/remyaP12/labcycle_3sem/blob/main/5_labcycle2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Build a Convolutional Neural Network (CNN) to classify handwritten digits, and perform
hyperparameter tuning (learning rate, batch size etc.) for improved performance.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical

# Load dataset
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Normalize
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# Reshape for CNN
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

# One-hot encoding
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

def build_cnn(learning_rate=0.001):
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
        MaxPooling2D(2,2),

        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D(2,2),

        Flatten(),
        Dense(128, activation='relu'),
        Dense(10, activation='softmax')
    ])

    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

baseline_model = build_cnn(learning_rate=0.001)

history_baseline = baseline_model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2
)

baseline_acc = baseline_model.evaluate(X_test, y_test, verbose=0)[1]
print("Baseline Test Accuracy:", baseline_acc)

learning_rates = [0.01, 0.001, 0.0001]
lr_results = {}

for lr in learning_rates:
    model = build_cnn(learning_rate=lr)
    model.fit(
        X_train, y_train,
        epochs=3,
        batch_size=128,
        validation_split=0.2,
        verbose=0
    )
    acc = model.evaluate(X_test, y_test, verbose=0)[1]
    lr_results[lr] = acc

print(lr_results)

batch_sizes = [32, 64, 128]
batch_results = {}

for bs in batch_sizes:
    model = build_cnn(learning_rate=0.001)
    model.fit(
        X_train, y_train,
        epochs=3,
        batch_size=bs,
        validation_split=0.2,
        verbose=0
    )
    acc = model.evaluate(X_test, y_test, verbose=0)[1]
    batch_results[bs] = acc

print(batch_results)

optimizers = {
    "Adam": tf.keras.optimizers.Adam(0.001),
    "SGD": tf.keras.optimizers.SGD(0.01, momentum=0.9)
}

opt_results = {}

for name, opt in optimizers.items():
    model = build_cnn()
    model.compile(
        optimizer=opt,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    model.fit(
        X_train, y_train,
        epochs=3,
        batch_size=64,
        validation_split=0.2,
        verbose=0
    )
    acc = model.evaluate(X_test, y_test, verbose=0)[1]
    opt_results[name] = acc

print(opt_results)

final_model = build_cnn(learning_rate=0.001)

final_model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2
)

final_acc = final_model.evaluate(X_test, y_test)[1]
print("Final Tuned Model Accuracy:", final_acc)

history = final_model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2
)

import matplotlib.pyplot as plt
plt.figure()
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('CNN Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(['Training Accuracy', 'Validation Accuracy'])
plt.show()

plt.figure()
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('CNN Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend(['Training Loss', 'Validation Loss'])
plt.show()

import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
y_pred = final_model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

cm = confusion_matrix(y_true, y_pred_classes)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=range(10)
)

disp.plot(cmap='Blues')
plt.title("Confusion Matrix - CNN MNIST")
plt.show()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.8469 - loss: 0.5181 - val_accuracy: 0.9759 - val_loss: 0.0799
Epoch 2/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9774 - loss: 0.0738 - val_accuracy: 0.9849 - val_loss: 0.0515
Epoch 3/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9858 - loss: 0.0438 - val_accuracy: 0.9858 - val_loss: 0.0486
Epoch 4/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9878 - loss: 0.0372 - val_accuracy: 0.9858 - val_loss: 0.0466
Epoch 5/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9919 - loss: 0.0264 - val_accuracy: 0.9877 - val_loss: 0.0438
Baseline Test Accuracy: 0.9879000186920166
{0.01: 0.9825000166893005, 0.001: 0.9846000075340271, 0.0001: 0.9715999960899353}
{32: 0.9861999750137329, 64: 0.9894999861717224, 128: 0.9876999855041504}
